In [12]:
# Only these two are needed — Open-Meteo needs no API key, no dotenv.
# !pip install requests pandas --quiet

In [13]:
# imports
import requests
import pandas as pd
from datetime import datetime, timezone
from datetime import date, timedelta
import numpy as np
import os

In [14]:
LAHORE = {"latitude": 31.5657, "longitude": 74.3142}
CITY = "Lahore"

AQI_URL      = "https://air-quality-api.open-meteo.com/v1/air-quality"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
ARCHIVE_URL  = "https://archive-api.open-meteo.com/v1/archive"

AQI_HOURLY = ("us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,"
              "sulphur_dioxide,ozone,dust,aerosol_optical_depth")
AQI_CURRENT = AQI_HOURLY + ",ammonia"

WX_HOURLY = ("temperature_2m,relative_humidity_2m,surface_pressure,"
             "wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m,precipitation")
WX_CURRENT = ("temperature_2m,relative_humidity_2m,surface_pressure,"
              "wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m")

## 1. Fetch functions

Four small functions: live vs historical, for air quality and weather.

In [15]:
def fetch_aqi_live():
    """Current air quality for Lahore (modelled, updates ~15-minutely)."""
    r = requests.get(AQI_URL, params={**LAHORE, "current": AQI_CURRENT, "timezone": "auto"})
    r.raise_for_status()
    return r.json()


def fetch_weather_live():
    """Current weather for Lahore."""
    r = requests.get(FORECAST_URL, params={**LAHORE, "current": WX_CURRENT, "timezone": "auto"})
    r.raise_for_status()
    return r.json()

def fetch_aqi_history(start_date, end_date):
    """Hourly air-quality history"""
    r = requests.get(AQI_URL, params={
        **LAHORE, "hourly": AQI_HOURLY,
        "start_date": start_date, "end_date": end_date, "timezone": "auto",
    })
    r.raise_for_status()
    return r.json()


def fetch_weather_history(start_date, end_date):
    """Hourly weather history from the ERA5 archive (note: ~5-day lag near today)."""
    r = requests.get(ARCHIVE_URL, params={
        **LAHORE, "hourly": WX_HOURLY,
        "start_date": start_date, "end_date": end_date, "timezone": "auto",
    })
    r.raise_for_status()
    return r.json()

## 2. Parse responses into DataFrames

Open-Meteo returns each variable as a parallel array aligned to a shared `time` array,
so an hourly block converts straight into a DataFrame.

In [16]:
def hourly_to_frame(resp):
    """Convert an Open-Meteo `hourly` block into a tidy DataFrame indexed by time."""
    block = resp["hourly"]
    df = pd.DataFrame(block)
    df["time"] = pd.to_datetime(df["time"])
    return df


def current_to_row(aqi_resp, wx_resp):
    """Merge the two `current` blocks into a single one-row DataFrame."""
    a = aqi_resp["current"]
    w = wx_resp["current"]
    row = {
        "time": pd.to_datetime(a["time"]),
        "city": CITY,
        # weather
        "temperature_2m": w.get("temperature_2m"),
        "relative_humidity_2m": w.get("relative_humidity_2m"),
        "surface_pressure": w.get("surface_pressure"),
        "wind_speed_10m": w.get("wind_speed_10m"),
        "wind_direction_10m": w.get("wind_direction_10m"),
        "cloud_cover": w.get("cloud_cover"),
        "dew_point_2m": w.get("dew_point_2m"),
        # air quality
        "us_aqi": a.get("us_aqi"),
        "pm2_5": a.get("pm2_5"),
        "pm10": a.get("pm10"),
        "carbon_monoxide": a.get("carbon_monoxide"),
        "nitrogen_dioxide": a.get("nitrogen_dioxide"),
        "sulphur_dioxide": a.get("sulphur_dioxide"),
        "ozone": a.get("ozone"),
        "dust": a.get("dust"),
        "aerosol_optical_depth": a.get("aerosol_optical_depth"),
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    return pd.DataFrame([row])

## 3. Feature engineering

- **Time-based:** hour, day, month, day of week, weekend flag.
- **Derived:** PM2.5/PM10 ratio, temp–humidity index, AQI category, AQI change rate, plus
  lag and rolling features that give a prediction model recent history to work with.

In [17]:
def add_time_features(df, time_col="time"):
    ts = pd.to_datetime(df[time_col])
    df["hour"] = ts.dt.hour
    df["day"] = ts.dt.day
    df["month"] = ts.dt.month
    df["day_of_week"] = ts.dt.dayofweek           # 0 = Monday
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    # cyclical encodings (help models treat hour 23 and 0 as adjacent)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_derived_features(df):
    # ratios / composite indices (single-row safe)
    df["pm25_pm10_ratio"] = df["pm2_5"] / df["pm10"]
    df["temp_humidity_index"] = df["temperature_2m"] * df["relative_humidity_2m"] / 100
    df["aqi_category"] = pd.cut(
        df["us_aqi"],
        bins=[-1, 50, 100, 150, 200, 300, 500],
        labels=["Good", "Moderate", "Unhealthy(SG)", "Unhealthy",
                "Very Unhealthy", "Hazardous"],
    )
    return df


def add_sequential_features(df):
    """Change rate, lags, and rolling stats. Needs a sorted, continuous time series."""
    df = df.sort_values("time").reset_index(drop=True)

    # AQI change rate (per hour). With hourly data the gap is ~1h.
    df["aqi_prev"] = df["us_aqi"].shift(1)
    hours_elapsed = df["time"].diff().dt.total_seconds() / 3600
    df["aqi_change_rate"] = (df["us_aqi"] - df["aqi_prev"]) / hours_elapsed

    # lag features
    for lag in (1, 3, 24):
        df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)
        df[f"pm25_lag_{lag}h"] = df["pm2_5"].shift(lag)

    # rolling stats (trailing window, shifted by 1 to avoid leaking the current value)
    df["aqi_roll_mean_24h"] = df["us_aqi"].shift(1).rolling(24, min_periods=6).mean()
    df["pm25_roll_mean_24h"] = df["pm2_5"].shift(1).rolling(24, min_periods=6).mean()
    df["aqi_roll_max_24h"] = df["us_aqi"].shift(1).rolling(24, min_periods=6).max()

    return df


def build_features(df):
    """Full feature pipeline for a (multi-row) time series."""
    df = df.copy()
    df["city"] = CITY
    df = add_time_features(df)
    df = add_derived_features(df)
    df = add_sequential_features(df)
    return df

## 4. Historical backfill

Pull the full history once, merge weather + air quality on `time`, and run the feature
pipeline to produce the training dataset.

In [18]:
BACKFILL_END   = (date.today() - timedelta(days=5)).isoformat()   # ERA5 lag buffer
BACKFILL_START = (date.today() - timedelta(days=730)).isoformat() # 2 years; use 180 for 6 months 

aqi_hist = hourly_to_frame(fetch_aqi_history(BACKFILL_START, BACKFILL_END))
wx_hist  = hourly_to_frame(fetch_weather_history(BACKFILL_START, BACKFILL_END))

# inner join keeps only hours present in both series
raw_hist = wx_hist.merge(aqi_hist, on="time", how="inner")
print("merged raw rows:", len(raw_hist))
raw_hist.head()

merged raw rows: 17424


,time,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m,precipitation,us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,aerosol_optical_depth
0,2024-09-06 00:00:00,26.2,91,977.6,6.9,129,10,24.6,0.0,120,54.8,78.5,851.0,34.9,15.0,63.0,0.0,0.83
1,2024-09-06 01:00:00,26.0,91,977.5,6.8,148,9,24.4,0.0,120,44.9,64.3,668.0,24.2,11.4,67.0,0.0,0.91
2,2024-09-06 02:00:00,25.8,91,977.2,6.0,155,0,24.3,0.0,120,35.9,51.5,517.0,15.7,8.8,74.0,0.0,1.09
3,2024-09-06 03:00:00,25.6,93,977.1,5.9,166,7,24.3,0.0,120,25.9,37.3,427.0,10.8,8.1,85.0,0.0,1.12
4,2024-09-06 04:00:00,23.5,95,978.7,21.3,19,99,22.6,0.7,118,18.7,27.0,368.0,8.1,8.4,99.0,0.0,0.94


In [19]:
features_hist = build_features(raw_hist)
print("feature rows:", len(features_hist), "| columns:", len(features_hist.columns))
features_hist.tail()

feature rows: 17424 | columns: 42


,time,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m,precipitation,us_aqi,...,aqi_change_rate,aqi_lag_1h,pm25_lag_1h,aqi_lag_3h,pm25_lag_3h,aqi_lag_24h,pm25_lag_24h,aqi_roll_mean_24h,pm25_roll_mean_24h,aqi_roll_max_24h
17419,2026-09-01 19:00:00,33.8,53,979.4,4.1,135,7,22.9,0.0,181,...,-5.0,186.0,52.3,161.0,61.7,154.0,54.4,157.166667,63.775000,186.0
17420,2026-09-01 20:00:00,32.9,59,980.2,3.6,123,6,23.9,0.0,166,...,-15.0,181.0,56.3,180.0,49.8,154.0,59.3,158.291667,63.854167,186.0
17421,2026-09-01 21:00:00,32.1,63,980.8,3.6,81,11,24.3,0.0,156,...,-10.0,166.0,56.3,186.0,52.3,154.0,62.6,158.791667,63.729167,186.0
17422,2026-09-01 22:00:00,31.4,66,980.9,4.7,72,12,24.3,0.0,156,...,0.0,156.0,56.7,181.0,56.3,154.0,64.5,158.875000,63.483333,186.0
17423,2026-09-01 23:00:00,31.9,66,980.7,6.4,170,22,24.7,0.0,155,...,-1.0,156.0,56.1,166.0,56.3,154.0,65.7,158.958333,63.133333,186.0


In [20]:
# Rows before the largest lag/rolling window won't have full sequential features.
# Drop the warm-up period so the training set has no partial-history rows.
train_df = features_hist.dropna(subset=["aqi_lag_24h", "aqi_roll_mean_24h"]).reset_index(drop=True)
print("training rows after warm-up drop:", len(train_df))
train_df.head()

training rows after warm-up drop: 17400


,time,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m,precipitation,us_aqi,...,aqi_change_rate,aqi_lag_1h,pm25_lag_1h,aqi_lag_3h,pm25_lag_3h,aqi_lag_24h,pm25_lag_24h,aqi_roll_mean_24h,pm25_roll_mean_24h,aqi_roll_max_24h
0,2024-09-07 00:00:00,26.3,88,980.3,11.7,108,3,24.2,0.0,87,...,-3.0,90.0,25.7,129.0,25.1,120.0,54.8,117.791667,28.800000,150.0
1,2024-09-07 01:00:00,25.9,90,980.2,10.0,111,0,24.1,0.0,85,...,-2.0,87.0,25.0,109.0,25.1,120.0,44.9,116.416667,27.558333,150.0
2,2024-09-07 02:00:00,25.5,93,980.4,8.6,105,3,24.3,0.0,83,...,-2.0,85.0,24.7,90.0,25.7,120.0,35.9,114.958333,26.716667,150.0
3,2024-09-07 03:00:00,25.3,95,980.3,7.6,115,3,24.4,0.0,83,...,0.0,83.0,24.9,87.0,25.0,120.0,25.9,113.416667,26.258333,150.0
4,2024-09-07 04:00:00,25.1,96,980.3,6.2,126,17,24.4,0.0,83,...,0.0,83.0,25.1,85.0,24.7,118.0,18.7,111.875000,26.225000,150.0


## 5. Write to the Feature Store (Hopsworks)

For a single-city hourly series, `event_time="time"` gives you time-travel / point-in-time
joins; `primary_key=["city"]` plus the event time identifies each row.

In [21]:
# !pip install "hopsworks[python]"
!pip install "hopsworks[python,delta]"


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import hopsworks

project = hopsworks.login(
    project="project123456789",              # your actual project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_KEY"]
)

fs = project.get_feature_store()

aqi_fg = fs.get_or_create_feature_group(
    name="lahore_aqi_features",
    version=1,
    primary_key=["city"],
    event_time="time",
    description="Open-Meteo weather + pollutant features for Lahore (hourly)",
    online_enabled=True,
    time_travel_format="HUDI",
)

train_df["aqi_category"] = train_df["aqi_category"].astype(str).replace("nan", "Unknown")
aqi_fg.insert(train_df)

2026-09-06 13:26:31,850 INFO: Closing external client and cleaning up certificates.
2026-09-06 13:26:31,855 INFO: Connection closed.
2026-09-06 13:26:31,861 INFO: Initializing external client
2026-09-06 13:26:31,862 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-06 13:26:34,182 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/43139


Uploading Dataframe: 100.00% |██████████| Rows 17400/17400 | Elapsed Time: 00:06 | Remaining Time: 00:00


Launching job: lahore_aqi_features_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/43139/jobs/named/lahore_aqi_features_1_offline_fg_materialization/executions


(Job('lahore_aqi_features_1_offline_fg_materialization', 'SPARK'), None)

## 6. Gap-fill: close the ERA5 lag so "now" is actually now

The 2-year backfill above stops 5 days back because ERA5 archive weather isn't published any
sooner. This step re-fetches just the missing days from Open-Meteo's **live forecast model**
(no archive lag) for both weather and air quality, stitches them onto the tail of the history
already in memory (so the 24h lag/rolling features are seeded correctly), and inserts only the
new rows — bringing the feature store's latest row up to the current hour.

Run this cell (after the one above) any time you want the dashboard to reflect today instead of
`BACKFILL_END`. Once the hourly CI/CD pipeline exists, this becomes that pipeline's job.

In [23]:
gap_start = (pd.to_datetime(BACKFILL_END) + timedelta(days=1)).date().isoformat()
gap_end   = date.today().isoformat()

# FORECAST_URL (not the ERA5 archive) also serves recent-past hourly weather
# with no lag, which is exactly the window ERA5 hasn't published yet.
wx_gap = hourly_to_frame(requests.get(FORECAST_URL, params={
    **LAHORE, "hourly": WX_HOURLY,
    "start_date": gap_start, "end_date": gap_end, "timezone": "auto",
}).json())
aqi_gap = hourly_to_frame(fetch_aqi_history(gap_start, gap_end))

raw_gap = wx_gap.merge(aqi_gap, on="time", how="inner")
print(f"gap-fill rows fetched ({gap_start} -> {gap_end}):", len(raw_gap))

# stitch onto the tail of the 2-year history so lag/rolling features are
# seeded correctly at the seam, then keep only the genuinely new rows
raw_seam = pd.concat([raw_hist.tail(48), raw_gap], ignore_index=True) \
             .drop_duplicates("time").sort_values("time").reset_index(drop=True)
features_gap = build_features(raw_seam)

# drop rows the API padded with same-day forecast hours that haven't happened yet
now_pkt = pd.Timestamp.utcnow().tz_localize(None) + pd.Timedelta(hours=5)
features_gap = features_gap[
    (features_gap["time"] >= pd.to_datetime(gap_start)) & (features_gap["time"] <= now_pkt)
].reset_index(drop=True)
features_gap["aqi_category"] = features_gap["aqi_category"].astype(str).replace("nan", "Unknown")

print("new rows to insert:", len(features_gap), "| latest time:", features_gap["time"].max())
aqi_fg.insert(features_gap)

gap-fill rows fetched (2026-09-02 -> 2026-09-06): 120
new rows to insert: 110 | latest time: 2026-09-06 13:00:00


Uploading Dataframe: 100.00% |██████████| Rows 110/110 | Elapsed Time: 00:01 | Remaining Time: 00:00
Use fg.materialization_job.run(args=-op offline_fg_materialization -path hdfs:///Projects/project123456789/Resources/jobs/lahore_aqi_features_1_offline_fg_materialization/config_1788032378737) to trigger the materialization job again.


(Job('lahore_aqi_features_1_offline_fg_materialization', 'SPARK'), None)